<a href="https://colab.research.google.com/github/shikatest/rlhf-tuning-1/blob/main/Generalized_Agents_Colabs/Other_Tools_and_Services/444_base/Agent-444_base-Merged.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Sample ID**: 444




**Query**: Mark the Jira issue in project "DevTools" with the summary "Refactor Logging System" as "In Review" and assign it to a user named "Sarah" with email address: sarah@atlassian.com. Then, update the Google Calendar event titled "Log Refactor Status Check" in the "Engineering Calendar", originally scheduled for April 22, 2025, 4:00?5:00 PM, by renaming it to "Log Refactor Review Meeting" and changing the time to April 23, 2025, 3:00?4:00 PM. Finally, post a message in the Slack channel "infra-updates" saying: ?The logging refactor task has been moved to review, and the meeting has been rescheduled to April 23, 3?4 PM.?




**DB Type**: Base Case




**Case Description**: The Jira project "DevTools" exists and contains the issue "Refactor Logging System". The issue is currently not assigned to a user named "Sarah" with email sarah@atlassian.com and not in "In Review" status.
The issue is to be updated with the new status and assignee.
The calendar "Engineering Calendar" exists and contains the event "Log Refactor Status Check" scheduled on April 22, 2025, 4:00?5:00 PM.
The event is to be renamed to "Log Refactor Review Meeting" and rescheduled to April 23, 2025, 3:00?4:00 PM.
The Slack channel "infra-updates" exists and does not contain a message with the specified text. A new message is to be posted.




**Global/Context Variables**:


- jira_project_name = "DevTools"
- jira_issue_summary = "Refactor Logging System"
- jira_user_email = "sarah@atlassian.com"
- jira_user_name ="Sarah"
- jira_expected_status = "In Review"
- calendar_name = "Engineering Calendar"
- original_event_summary = "Log Refactor Status Check"
- updated_event_summary = "Log Refactor Review Meeting"
- original_start = "2025-04-22T16:00:00Z"
- original_end   = "2025-04-22T17:00:00Z"
- updated_start  = "2025-04-23T15:00:00Z"
- updated_end    = "2025-04-23T16:00:00Z"
- slack_channel_name = "infra-updates"
- slack_message_text =
    "The logging refactor task has been moved to review, and the meeting has been rescheduled to April 23, 3?4 PM."







**Datetime Context Variables**:
- user_timezone = "UTC"

**APIs**:

- jira
- google_calendar
- slack


# Set Up

## Download relevant files

In [1]:
import io
import os
import sys
import zipfile
import shutil
import re
from google.colab import auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload

# Version to download
VERSION = "0.1.5"  # Version of the API

# Define paths
CONTENT_DIR = '/content'
APIS_DIR = os.path.join(CONTENT_DIR, 'APIs')
DBS_DIR = os.path.join(CONTENT_DIR, 'DBs')
SCRIPTS_DIR = os.path.join(CONTENT_DIR, 'Scripts')
FC_DIR = os.path.join(CONTENT_DIR, 'Schemas')
ZIP_PATH = os.path.join(CONTENT_DIR, f'APIs_V{VERSION}.zip')

# Google Drive Folder ID where versioned APIs zip files are stored
APIS_FOLDER_ID = '1QpkAZxXhVFzIbm8qPGPRP1YqXEvJ4uD4'

# List of items to extract from the zip file
ITEMS_TO_EXTRACT = ['APIs/', 'DBs/', 'Scripts/', 'Schemas/']

# Clean up existing directories and files
for path in [APIS_DIR, DBS_DIR, SCRIPTS_DIR, FC_DIR, ZIP_PATH]:
    if os.path.exists(path):
        if os.path.isdir(path):
            shutil.rmtree(path)
        else:
            os.remove(path)

# Authenticate and create the drive service
auth.authenticate_user()
drive_service = build('drive', 'v3')

# Helper function to download a file from Google Drive
def download_drive_file(service, file_id, output_path, file_name=None, show_progress=True):
    """Downloads a file from Google Drive"""
    destination = output_path
    request = service.files().get_media(fileId=file_id)
    with io.FileIO(destination, 'wb') as fh:
        downloader = MediaIoBaseDownload(fh, request)
        done = False
        while not done:
            status, done = downloader.next_chunk()
            if show_progress:
                print(f"Download progress: {int(status.progress() * 100)}%")


# 1. List files in the specified APIs folder
print(f"Searching for APIs zip file with version {VERSION} in folder: {APIS_FOLDER_ID}...")
apis_file_id = None

try:
    query = f"'{APIS_FOLDER_ID}' in parents and trashed=false"
    results = drive_service.files().list(q=query, fields="files(id, name)").execute()
    files = results.get('files', [])
    for file in files:
        file_name = file.get('name', '')
        if file_name.lower() == f'apis_v{VERSION.lower()}.zip':
            apis_file_id = file.get('id')
            print(f"Found matching file: {file_name} (ID: {apis_file_id})")
            break

except Exception as e:
    print(f"An error occurred while listing files in Google Drive: {e}")

if not apis_file_id:
    print(f"Error: Could not find APIs zip file with version {VERSION} in the specified folder.")
    sys.exit("Required APIs zip file not found.")

# 2. Download the found APIs zip file
print(f"Downloading APIs zip file with ID: {apis_file_id}...")
download_drive_file(drive_service, apis_file_id, ZIP_PATH, file_name=f'APIs_V{VERSION}.zip')

# 3. Extract specific items from the zip file to /content
print(f"Extracting specific items from {ZIP_PATH} to {CONTENT_DIR}...")
try:
    with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
        zip_contents = zip_ref.namelist()

        for member in zip_contents:
            extracted = False
            for item_prefix in ITEMS_TO_EXTRACT:
              if member == item_prefix or member.startswith(item_prefix):
                    zip_ref.extract(member, CONTENT_DIR)
                    extracted = True
                    break

except zipfile.BadZipFile:
    print(f"Error: The downloaded file at {ZIP_PATH} is not a valid zip file.")
    sys.exit("Invalid zip file downloaded.")
except Exception as e:
    print(f"An error occurred during extraction: {e}")
    sys.exit("Extraction failed.")


# 4. Clean up
if os.path.exists(ZIP_PATH):
    os.remove(ZIP_PATH)

# 5. Add APIs to path
if os.path.exists(APIS_DIR):
    sys.path.append(APIS_DIR)
else:
    print(f"Error: APIS directory not found at {APIS_DIR} after extraction. Cannot add to path.")

# 6. Quick verification
# Check for the presence of the extracted items
verification_paths = [APIS_DIR, DBS_DIR, SCRIPTS_DIR]
all_present = True
print("\nVerifying extracted items:")
for path in verification_paths:
    if os.path.exists(path):
        print(f"? {path} is present.")
    else:
        print(f"? {path} is MISSING!")
        all_present = False

if all_present:
    print(f"\n? Setup complete! Required items extracted to {CONTENT_DIR}.")
else:
    print("\n? Setup failed! Not all required items were extracted.")

# 7. Generate Schemas

print("\nGenerating FC Schemas")

# Change working directory to the source folder

# Iterate through the packages in the /content/APIs directory

    # Check if it's a directory (to avoid processing files)
        # Call the function to generate schema for the current package
print(f"? Successfully generated {len(os.listdir(FC_DIR))} FC Schemas to {FC_DIR}")
os.chdir(CONTENT_DIR)

MessageError: Error: credential propagation was unsuccessful

## Install Dependencies and Clone Repositories

In [ ]:
!pip install -r /content/APIs/requirements.txt

## Import APIs and initiate DBs

In [ ]:
# proto_ignore
import random
import sys
import uuid
import secrets

# Import libraries to ensure all initializations by the python libraries are complete
import google_calendar
import jira


def patch_randomness(seed=42):
    rng = random.Random(seed)
    random.seed(seed)

    # Patch uuid.uuid4
    def deterministic_uuid4():
        return uuid.UUID(int=rng.getrandbits(128))
    sys.modules['uuid'].uuid4 = deterministic_uuid4

    # Patch secrets to use the same deterministic random generator
    class DeterministicRandom:
        def randbelow(self, n):
            return rng.randrange(n)

        def choice(self, seq):
            return rng.choice(seq)

        def randbits(self, k):
            return rng.getrandbits(k)

        def randint(self, a, b):
            return rng.randint(a, b)
    sys.modules['secrets'] = DeterministicRandom()

patch_randomness()

In [ ]:
import jira
import google_calendar
import slack

# Load the databases
jira.SimulationEngine.db.load_state("/content/DBs/JiraDefaultDB.json")
google_calendar.SimulationEngine.db.load_state("/content/DBs/CalendarDefaultDB.json")
slack.SimulationEngine.db.load_state("/content/DBs/SlackDefaultDB.json")


# --- Constants ---
jira_project_name = "DevTools"
jira_project_key = "DEVTOOLS"
jira_issue_summary = "Refactor Logging System"
jira_user_email = "sarah@atlassian.com"
jira_status = "Open"
jira_expected_status = "In Review"
calendar_name = "Engineering Calendar"
original_event_summary = "Log Refactor Status Check"
updated_event_summary = "Log Refactor Review Meeting"
original_start = "2025-04-22T16:00:00Z"
original_end   = "2025-04-22T17:00:00Z"
updated_start  = "2025-04-23T15:00:00Z"
updated_end    = "2025-04-23T16:00:00Z"
slack_channel_name = "infra-updates"
slack_message_text ="The logging refactor task has been moved to review, and the meeting has been rescheduled to April 23, 3?4 PM."

# Define local variables
jira_user_name = "Sarah"

# Create the Jira project.
jira.create_project(
    proj_key=jira_project_key,
    proj_name=jira_project_name
)
print(f"Created Jira project: {jira_project_name}")

# Create the Jira issue.
issue_fields = {
    "project": jira_project_key,
    "summary": jira_issue_summary,
    "status": jira_status,
    "assignee": {"name": "unassigned"}
}
created_issue = jira.create_issue(fields=issue_fields)
created_issue_id = created_issue.get('id')
print(f"Created Jira issue: {created_issue_id} with summary '{jira_issue_summary}'")

# Optionally, fetch and print issue details to verify stored project key and summary.
created_issue_details = jira.get_issue_by_id(issue_id=created_issue_id)
print("Created issue details:", created_issue_details)

# Create the Jira user.
jira.create_user({
    "name": jira_user_name,
    "emailAddress": jira_user_email,
    "displayName": "Sarah"
})
print(f"Created Jira user: {jira_user_name}")

# --- Google Calendar Setup ---
calendar_resource = {"summary": calendar_name}
calendar_response = google_calendar.create_secondary_calendar(resource=calendar_resource)
print(f"Created Google Calendar: {calendar_name}")

google_calendar.create_calendar_list_entry(
    resource={
        "id": calendar_response["id"],
        "summary": calendar_name
    }
)
print(f"Added calendar '{calendar_name}' to calendar list")

# --- Create Initial Calendar Event ---
event_payload = {
    "summary": original_event_summary,
    "start": {"dateTime": original_start, "timeZone": "UTC"},
    "end": {"dateTime": original_end, "timeZone": "UTC"}
}
new_event = google_calendar.create_event(
    calendarId=calendar_response["id"],
    resource=event_payload
)
print(f"Created calendar event: {original_event_summary} with ID {new_event['id']}")

# --- Slack Setup ---
slack.create_channel(name=slack_channel_name)
print(f"Created Slack channel: {slack_channel_name}")


# --- Autofix: standardize calendars' timezone to UTC ---
def set_all_calendars_timezone_to_utc():
    """
    Set timeZone='UTC' for every calendar, preserving summary/description.
    Skips calendars already in UTC.
    """
    cl = google_calendar.list_calendar_list_entries()
    items = cl.get("items", [])
    results = []

    for cal in items:
        cal_id = cal.get("id")
        summary = cal.get("summary", "")
        description = cal.get("description", "")

        resource = {
            "summary": summary,
            "description": description,
            "timeZone": "UTC",
        }

        updated = google_calendar.update_calendar_metadata(
                calendarId=cal_id,
                resource=resource
            )

        results.append(updated)

    return results

# Apply after DBs are initialized
set_all_calendars_timezone_to_utc()


# Initial Assertion
1. Assert that the Jira project "DevTools" exists.
2. Assert that user with email address sarah@atlassian.com exists in Jira.
3. Assert that the name of this user is "Sarah".
4. Assert that exactly one Jira issue with the summary "Refactor Logging System" exists in the "DevTools" project.
5. Assert that Jira issue has details present and it is not None.
6. Assert that the above issue is not assigned to user with email address sarah@atlassian.com
7. Assert that status of above issue is not "In Review".
8. Assert that the Google Calendar titled "Engineering Calendar" exists.
9. Assert that exactly one event titled "Log Refactor Status Check" is scheduled on April 22, 2025, from 4:00 PM to 5:00 PM in the "Engineering Calendar".
10. Assert that no event titled "Log Refactor Review Meeting" exists in  "Engineering Calendar".
11. Assert that the Slack channel "infra-updates" exists.
12. Assert that the Slack channel "infra-updates" does not contain the message: "The logging refactor task has been moved to review, and the meeting has been rescheduled to April 23, 3?4 PM."

In [ ]:
from Scripts.assertions_utils import *
import jira
import google_calendar
import slack
import dateutil

# --- Constants ---
jira_project_name = "DevTools"
jira_issue_summary = "Refactor Logging System"
jira_user_email = "sarah@atlassian.com"
jira_user_name = "Sarah"
jira_expected_status = "In Review"
calendar_name = "Engineering Calendar"
original_event_summary = "Log Refactor Status Check"
updated_event_summary = "Log Refactor Review Meeting"
original_start = "2025-04-22T16:00:00Z"
original_end   = "2025-04-22T17:00:00Z"
updated_start  = "2025-04-23T15:00:00Z"
updated_end    = "2025-04-23T16:00:00Z"
slack_channel_name = "infra-updates"
slack_message_text = "The logging refactor task has been moved to review, and the meeting has been rescheduled to April 23, 3?4 PM."

# --- Begin Jira Assertions ---

# 1. Assert that the Jira project "DevTools" exists.
jira_project_key = next(
    (proj.get("key") for proj in jira.get_all_projects().get("projects", []) if compare_strings(proj.get("name", ""), jira_project_name)),
    None
)
project_response = jira.get_project_by_key(project_key=jira_project_key)
assert project_response is not None and project_response.get('key') == jira_project_key, (
    f"Initial Assertion Failed: Expected Jira project '{jira_project_key}' to exist, but it was not found or key mismatch. Found: {project_response}"
)

# 2. Assert that the user with email address sarah@atlassian.com exists in Jira.
user_response = jira.find_users(search_string=jira_user_email)
if user_response:
    user_response = user_response[0]

assert compare_strings(user_response.get('emailAddress', ""), jira_user_email), (
    f"Initial Assertion Failed: Expected Jira user with email '{jira_user_email}' to exist. Found: {user_response}"
)

assert compare_strings(user_response.get('displayName', ""), jira_user_name), (
    f"Initial Assertion Failed: Expected Jira user with display name '{jira_user_name}' to exist. Found: {user_response}"
)
jira_user_account_id = user_response.get('displayName')

# 3. Assert that exactly one Jira issue with the summary "Refactor Logging System" exists in the "DevTools" project.
jql = f'project = "{jira_project_key}" AND summary ~ "{jira_issue_summary}"'
search_response = jira.search_issues_jql(jql=jql)

matching_issues = [i for i in search_response.get("issues", []) if compare_strings(i.get("fields", {}).get("summary", ""), jira_issue_summary)]

assert len(matching_issues) == 1, (
    f"Initial Assertion Failed: Expected exactly one Jira issue with summary '{jira_issue_summary}' in project '{jira_project_key}', but found {len(matching_issues)}."
)

initial_issue_id = matching_issues[0].get("id")

# 4. Assert that Jira issue has details present and it is not None.
issue_details_response = jira.get_issue_by_id(issue_id=initial_issue_id)
assert issue_details_response is not None and 'fields' in issue_details_response, (
    f"Initial Assertion Failed: Could not fetch details for Jira issue ID '{initial_issue_id}'. Response: {issue_details_response}"
)
initial_issue_fields = issue_details_response.get("fields")

# 5. Assert that the above issue is not assigned to user with email address sarah@atlassian.com.
initial_assignee = initial_issue_fields.get('assignee')
initial_assignee_account_name = initial_assignee.get('name') if initial_assignee else ""
assert not compare_strings(initial_assignee_account_name, jira_user_account_id), (
    f"Initial Assertion Failed: Expected Jira issue '{initial_issue_id}' not to be assigned to user '{jira_user_email}', but it is. Assignee: {initial_assignee}"
)

# 6. Assert that the status of the above issue is not "In Review".
status_value = initial_issue_fields.get('status', "")
assert not compare_strings(status_value, jira_expected_status), (
    f"Initial Assertion Failed: Expected Jira issue '{initial_issue_id}' status not to be '{jira_expected_status}', but it is. Status: {status_value}"
)

# # --- Google Calendar Assertions ---

# 7. Assert that the Google Calendar titled "Engineering Calendar" exists.
calendar_list_response = google_calendar.list_calendar_list_entries()
engineering_calendar = next(
    (cal for cal in calendar_list_response.get('items', []) if compare_strings(cal.get('summary', ''), calendar_name)),
    {}
)
calendar_id = engineering_calendar.get('id')
assert calendar_id is not None, (
    f"Initial Assertion Failed: Expected to find Google Calendar '{calendar_name}' with a valid ID, but it was either not found or is missing an ID. Calendar data: {engineering_calendar}"
)
# 8. Assert that exactly one event titled "Log Refactor Status Check" is scheduled on April 22, 2025, from 4:00 PM to 5:00 PM
events_response = google_calendar.list_events(calendarId=calendar_id, timeMin=original_start, timeMax=original_end)

initial_matching_events = [
    event for event in events_response.get('items', [])
    if compare_strings(event.get('summary', ''), original_event_summary)
    # ✅ Add null check for dateTime
    and event.get('start', {}).get('dateTime') is not None
    and event.get('end', {}).get('dateTime') is not None
    and compare_datetimes(
        dateutil.parser.parse(event.get('start', {}).get('dateTime')),
        dateutil.parser.parse(original_start)
    )
    and compare_datetimes(
        dateutil.parser.parse(event.get('end', {}).get('dateTime')),
        dateutil.parser.parse(original_end)
    )
]
assert len(initial_matching_events) == 1, (
    f"Initial Assertion Failed: Expected exactly one event titled '{original_event_summary}' from {original_start} to {original_end} in calendar '{calendar_name}', but found {len(initial_matching_events)}. Found: {initial_matching_events}"
)
initial_event_id = initial_matching_events[0].get('id')


# 9. Assert that no event titled "Log Refactor Review Meeting" exists in "Engineering Calendar"
events_response = google_calendar.list_events(calendarId=calendar_id, timeMin=updated_start, timeMax=updated_end)
matching_updated_events = [
    event for event in events_response.get('items', [])
    if compare_strings(event.get('summary', ''), updated_event_summary)
]
assert len(matching_updated_events) == 0, (
    f"Initial Assertion Failed: Expected no event titled '{updated_event_summary}' in calendar '{calendar_name}' between {updated_start} and {updated_end}, but found {len(matching_updated_events)}. Found: {matching_updated_events}"
)

# --- Slack Assertions ---
# --- Slack Assertions ---

# 10. Assert that the Slack channel "infra-updates" exists.
channels_response = slack.list_channels(types="public_channel,private_channel") or {}
channels = channels_response.get('channels', []) or []

infra_updates_channel = next(
    (ch for ch in channels if compare_strings(ch.get('name', ''), slack_channel_name)),
    None
)

# Ensure channel exists before using its id
assert infra_updates_channel is not None and infra_updates_channel.get('id'), (
    f"Initial Assertion Failed: Slack channel '{slack_channel_name}' not found."
)

slack_channel_id = infra_updates_channel.get('id')

# 11. Assert that the Slack channel "infra-updates" does not contain the message
history_response = slack.get_conversation_history(channel=slack_channel_id, limit=200) or {}
messages = history_response.get('messages', []) or []
message_texts = [msg.get('text', '') for msg in messages]

# Avoid ValueError from compare_is_list_subset on empty list
if not message_texts:
    message_found = False
else:
    # You can keep the utility:
    message_found = compare_is_list_subset(slack_message_text, message_texts, list_comparison_function='any')
    # Or use a direct normalized substring check:
    # message_found = any(compare_is_string_subset(slack_message_text, t) for t in message_texts)

assert not message_found, (
    f"Initial Assertion Failed: Expected Slack channel '{slack_channel_name}' not to contain the message "
    f"'{slack_message_text}', but it was found."
)

# Action

- Mark the Jira issue in project "DevTools" with the summary "Refactor Logging System" as "In Review" and assign it to user with email address: sarah@atlassian.com.

- Then, update the Google Calendar event titled "Log Refactor Status Check" in the "Engineering Calendar", originally scheduled for April 22, 2025, 4:00?5:00 PM, by renaming it to "Log Refactor Review Meeting" and changing the time to April 23, 2025, 3:00?4:00 PM.

- Finally, post a message in the Slack channel "infra-updates" saying: ?The logging refactor task has been moved to review, and the meeting has been rescheduled to April 23, 3?4 PM.?

In [ ]:
# step 1
# proto_ignore
import jira
import google_calendar
import slack

In [ ]:
# step 2
jira.get_all_projects()

In [ ]:
# step 3
jira.search_issues_jql(
    jql='project = "DEVTOOLS" AND summary = "Refactor Logging System"'
)

In [ ]:
# step 4
issues = jira.search_issues_jql(jql='project = "DEVTOOLS" AND summary = "Refactor Logging System"')
issue_id = issues["issues"][0]["id"]

jira.update_issue_by_id(
    issue_id=issue_id,
    fields={"status": "In Review"}
)

In [ ]:
# step 5
jira.assign_issue_to_user(
    issue_id=issue_id,
    assignee={"name": "Sarah"}
)

In [ ]:
# step 6
google_calendar.list_calendar_list_entries()

In [ ]:
# step 7
calendar_list = google_calendar.list_calendar_list_entries()
calendar_id = None
for cal in calendar_list.get("items", []):
    if cal.get("summary") == "Engineering Calendar":
        calendar_id = cal.get("id")
        break

# ✅ Fix: robust event lookup, tolerate +00:00 or Z formats
event_id = None
events = google_calendar.list_events(calendarId=calendar_id)
for ev in events.get("items", []):
    summary = ev.get("summary", "")
    start_time = ev.get("start", {}).get("dateTime", "")
    if summary == "Log Refactor Status Check" and (
        start_time.startswith("2025-04-22T16:00:00")  # matches both Z and +00:00
    ):
        event_id = ev.get("id")
        break

if not event_id:
    print("⚠️ Could not find event 'Log Refactor Status Check'. Events found:", events.get("items", []))



In [ ]:
# step 8
if event_id:
    google_calendar.update_event(
        calendarId=calendar_id,
        eventId=event_id,
        resource={
            "summary": "Log Refactor Review Meeting",
            "start": {"dateTime": "2025-04-23T15:00:00Z", "timeZone": "UTC"},
            "end": {"dateTime": "2025-04-23T16:00:00Z", "timeZone": "UTC"}
        }
    )
    updated_event = google_calendar.get_event(calendarId=calendar_id, eventId=event_id)
    print("✅ Updated event verified:", updated_event)
else:
    raise ValueError("Event 'Log Refactor Status Check' not found — cannot update.")

In [ ]:
# step 9
slack.list_channels(types="public_channel,private_channel")


In [ ]:

# step 10
channels = slack.list_channels(types="public_channel,private_channel")
# ✅ Fix: select channel by name instead of first channel
channel_id = next(
    (ch.get("id") for ch in channels.get("channels", []) if ch.get("name") == slack_channel_name),
    None
)


if not channel_id:
    raise ValueError(f"Slack channel '{slack_channel_name}' not found.")

slack.post_chat_message(
    channel=channel_id,
    text="The logging refactor task has been moved to review, and the meeting has been rescheduled to April 23, 3–4 PM."
)

# Final Assertion

1. Assert that the issue with summary "Refactor Logging System" is assigned to user with email address sarah@atlassian.com
2. Assert that status of above issue is "In Review".
3. Assert that exactly one event titled "Log Refactor Review Meeting" is scheduled on April 23, 2025, from 3:00 PM to 4:00 PM in the "Engineering Calendar".
4. Assert that no event titled "Log Refactor Status Check" exists in "Engineering Calendar".
5. Assert that the Slack channel "infra-updates" contains exactly one message: "The logging refactor task has been moved to review, and the meeting has been rescheduled to April 23, 3?4 PM."

In [ ]:
from Scripts.assertions_utils import *
import jira
import google_calendar
from datetime import datetime

# --- Constants (post-action expectations) ---
jira_project_name     = "DevTools"
jira_issue_summary    = "Refactor Logging System"
jira_user_name        = "Sarah"
jira_expected_status  = "In Review"

calendar_name         = "Engineering Calendar"
original_event_summary= "Log Refactor Status Check"
updated_event_summary = "Log Refactor Review Meeting"
original_start        = "2025-04-22T16:00:00Z"
original_end          = "2025-04-22T17:00:00Z"
updated_start         = "2025-04-23T15:00:00Z"
updated_end           = "2025-04-23T16:00:00Z"

# =========================
#        JIRA CHECKS
# =========================

# Find project key safely
projects_resp   = jira.get_all_projects() or {}
projects        = projects_resp.get("projects", []) or []
jira_project_key= next((p.get("key") for p in projects if compare_strings(p.get("name",""), jira_project_name)), None)
assert jira_project_key, (
    f"Final Assertion Failed: Jira project '{jira_project_name}' not found. "
    f"Seen: {[p.get('name') for p in projects][:5]}"
)

# Find the issue by exact summary in that project
jql = f'project = "{jira_project_key}" AND summary = "{jira_issue_summary}"'
search_resp = jira.search_issues_jql(jql=jql) or {}
issues      = search_resp.get("issues", []) or []
issue       = next((i for i in issues if compare_strings((i.get("fields") or {}).get("summary",""), jira_issue_summary)), None)
assert issue is not None, (
    f"Final Assertion Failed: Jira issue with summary '{jira_issue_summary}' in '{jira_project_key}' not found."
)

final_issue_id      = issue.get("id")
issue_details_resp  = jira.get_issue_by_id(issue_id=final_issue_id) or {}
final_issue_fields  = issue_details_resp.get("fields", {}) or {}
assert final_issue_fields, (
    f"Final Assertion Failed: Could not fetch fields for Jira issue '{final_issue_id}'. Resp: {issue_details_resp}"
)

# 1) Issue assigned to expected user (by display name preferred, fallback to name)
final_assignee      = final_issue_fields.get("assignee") or {}
final_assignee_name = final_assignee.get("displayName") or final_assignee.get("name") or ""
assert final_assignee_name and compare_strings(final_assignee_name, jira_user_name), (
    f"Final Assertion Failed: Expected Jira issue '{final_issue_id}' assigned to '{jira_user_name}', "
    f"but found assignee: {final_assignee}"
)

# 2) Status is "In Review"
final_status = final_issue_fields.get("status", "")
assert compare_strings(final_status, jira_expected_status), (
    f"Final Assertion Failed: Expected Jira issue '{final_issue_id}' status '{jira_expected_status}', "
    f"but found '{final_status}'. Fields: {final_issue_fields}"
)

# =========================
#    GOOGLE CALENDAR
# =========================

# Locate the calendar
cal_list = (google_calendar.list_calendar_list_entries() or {}).get("items", []) or []
calendar_id = next((c.get("id") for c in cal_list if compare_strings(c.get("summary",""), calendar_name)), None)
assert calendar_id, (
    f"Final Assertion Failed: Calendar '{calendar_name}' not found. "
    f"Seen: {[c.get('summary') for c in cal_list][:5]}"
)

# 3) Exactly one updated event exists at new time with new title
events_updated = google_calendar.list_events(calendarId=calendar_id, timeMin=updated_start, timeMax=updated_end) or {}
matching_updated_events = []
for e in (events_updated.get("items", []) or []):
    ev_sum   = e.get("summary","")
    ev_start = (e.get("start",{}) or {}).get("dateTime")
    ev_end   = (e.get("end",  {}) or {}).get("dateTime")

    try:
        start_ok = ev_start and compare_datetimes(
            datetime.fromisoformat(ev_start.replace("Z","+00:00")),
            datetime.fromisoformat(updated_start.replace("Z","+00:00"))
        )
        end_ok   = ev_end and compare_datetimes(
            datetime.fromisoformat(ev_end.replace("Z","+00:00")),
            datetime.fromisoformat(updated_end.replace("Z","+00:00"))
        )
    except Exception:
        start_ok = end_ok = False

    if compare_strings(ev_sum, updated_event_summary) and start_ok and end_ok:
        matching_updated_events.append(e)

assert len(matching_updated_events) == 1, (
    f"Final Assertion Failed: Expected exactly one '{updated_event_summary}' from {updated_start} to {updated_end} "
    f"in '{calendar_name}', but found {len(matching_updated_events)}. Found: {matching_updated_events}"
)

# 4) Original event no longer present at old time window
events_original = google_calendar.list_events(calendarId=calendar_id, timeMin=original_start, timeMax=original_end) or {}
still_original = [
    e for e in (events_original.get("items", []) or [])
    if compare_strings(e.get("summary",""), original_event_summary)
]
assert len(still_original) == 0, (
    f"Final Assertion Failed: Original event '{original_event_summary}' still present between "
    f"{original_start} and {original_end}. Found: {still_original}"
)